# Reti neurali profonde

> **Nota AI Lab — ambiente di lettura.** Il testo seguente è la traduzione del notebook Kaggle originale conservato nel repository. I riferimenti a pulsanti, celle nascoste e invii descrivono l’ambiente Kaggle dell’originale; qui le celle di codice sono visibili. Per eseguire i notebook fuori da Kaggle, segui la guida AI Lab all’ambiente.



## Preparazione in Colab

1. Salvare una copia del notebook nel proprio Drive.
2. Eseguire la cella seguente e attendere «Preparazione completata».
3. Eseguire le celle nell’ordine e svolgere gli esercizi.

La cella di preparazione acquisisce i dataset e i controlli degli esercizi dal repository del corso e configura l’ambiente Python. Dopo il riavvio della sessione occorre eseguire nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '56f1300be5bf273a70dd4a30e704b20f642ec01dacc4d2d4b5a295a02e1da60f'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=True)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Introduzione

In questa lezione vedremo come costruire reti neurali capaci di apprendere le relazioni complesse per le quali le reti profonde sono famose.

L’idea fondamentale è la *modularità*: costruire una rete complessa a partire da unità funzionali più semplici. Abbiamo visto come un’unità lineare calcola una funzione lineare; ora vedremo come combinare e modificare queste singole unità per modellare relazioni più complesse.

# Strati

Le reti neurali organizzano normalmente i neuroni in **strati**, o *layer*. Riunendo unità lineari che condividono gli stessi ingressi, otteniamo uno strato **dense**.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/2MA4iMV.png" width="300" alt="Tre cerchi nello strato di ingresso collegati a due cerchi in uno strato dense.">
<figcaption><center>Uno strato dense di due unità lineari che ricevono due ingressi e un bias.</center></figcaption>
</figure>

Puoi pensare a ciascuno strato come a una trasformazione relativamente semplice. Attraverso una sequenza profonda di strati, la rete può trasformare gli ingressi in modi sempre più complessi. In una rete ben addestrata, ciascuno strato è una trasformazione che ci avvicina un po’ di più alla soluzione.

> **Molti tipi di strati**
>
> In Keras, «strato» è un concetto molto generale. Uno strato può essere essenzialmente qualsiasi tipo di *trasformazione dei dati*. Molti strati, come quelli [convoluzionali](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) e [ricorrenti](https://www.tensorflow.org/api_docs/python/tf/keras/layers/RNN), trasformano i dati usando neuroni e si distinguono soprattutto per lo schema delle connessioni. Altri servono per la [costruzione delle feature](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) o semplicemente per [operazioni aritmetiche](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Add). C’è un intero mondo di strati da scoprire: [esplorali](https://www.tensorflow.org/api_docs/python/tf/keras/layers)!

# La funzione di attivazione

Due strati dense senza nulla in mezzo, però, non sono migliori di un unico strato dense. Da soli, gli strati dense non possono portarci fuori dal mondo delle rette e dei piani. Ci serve qualcosa di *non lineare*: ci servono le funzioni di attivazione.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/OLSUEYT.png" width="400" alt="Relazioni lineari e non lineari a confronto.">
<figcaption><center>Senza funzioni di attivazione, le reti neurali possono apprendere soltanto relazioni lineari. Per adattarsi a delle curve occorrono funzioni di attivazione.</center></figcaption>
</figure>

Una **funzione di attivazione** è semplicemente una funzione applicata a ciascuna uscita di uno strato, cioè alle sue *attivazioni*. La più comune è la funzione *rettificatrice* $max(0, x)$.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/aeIyAlF.png" width="400" alt="Grafico della funzione rettificatrice: y=x per x>0 e y=0 per x<0, con una forma a gomito.">
</figure>

Il grafico della funzione rettificatrice è una retta la cui parte negativa è stata «rettificata» a zero. Applicandola alle uscite di un neurone, introduciamo una *piega* nei dati e ci allontaniamo dalle semplici rette.

Associando il rettificatore a un’unità lineare otteniamo una **rectified linear unit**, cioè un’unità lineare rettificata, o **ReLU**. Per questo è comune chiamare il rettificatore «funzione ReLU». Applicare l’attivazione ReLU a un’unità lineare significa che l’uscita diventa `max(0, w * x + b)`, rappresentabile così:

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/eFry7Yu.png" width="250" alt="Schema di una ReLU: simile a un’unità lineare, ma con un gomito al posto del simbolo più.">
<figcaption><center>Un’unità lineare rettificata.</center></figcaption>
</figure>


# Sovrapporre strati dense

Ora che abbiamo introdotto la non linearità, vediamo come sovrapporre gli strati per ottenere trasformazioni complesse dei dati.

<figure style="padding: 1em;">
<img src="https://storage.googleapis.com/kaggle-media/learn/images/Y5iwFQZ.png" width="450" alt="Uno strato di ingresso, due strati nascosti e uno strato lineare finale.">
<figcaption><center>Una sequenza di strati dense forma una rete «completamente connessa».</center></figcaption>
</figure>

Gli strati che precedono quello di uscita sono talvolta detti **nascosti**, perché non ne osserviamo mai direttamente le uscite.

Nota che lo strato finale, quello di uscita, è un’unità lineare: non ha quindi una funzione di attivazione. Questo rende la rete adatta a un compito di regressione, nel quale vogliamo prevedere un valore numerico arbitrario. Altri compiti, come la classificazione, possono richiedere una funzione di attivazione sull’uscita.

## Costruire modelli Sequential

Il modello `Sequential` che stiamo usando collega una lista di strati nell’ordine dal primo all’ultimo: il primo riceve gli ingressi, l’ultimo produce l’uscita. Il codice seguente crea il modello della figura:


In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    # Layer nascosti con attivazione ReLU.
    layers.Dense(units=4, activation='relu', input_shape=[2]),
    layers.Dense(units=3, activation='relu'),
    # Layer di output lineare.
    layers.Dense(units=1),
])

Assicurati di passare tutti gli strati insieme in una lista, come `[layer, layer, layer, ...]`, anziché come argomenti separati. Per aggiungere una funzione di attivazione a uno strato, basta specificarne il nome nell’argomento `activation`.

# Ora tocca a te

Ora [**crea una rete neurale profonda**](https://prof-davidaffo.github.io/kaggle-learntools/corsi/deep_learning_intro/ex2.html) per il dataset *Concrete*.
